In [4]:
import socket

# UDP_IP = "udp2dmx"  # IP-Adresse deines ESP32
# UDP_IP = "192.168.178.55"  # IP-Adresse deines ESP32
UDP_IP = "192.168.188.85"  # IP-Adresse deines ESP32
UDP_PORT = 6454

def send_dmx_command(command: str):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.sendto(command.encode(), (UDP_IP, UDP_PORT))
    print(f"Gesendet: {command}")

# Beispiele für Kommandos:

# # Direktwert 129 auf Kanal 2
# send_dmx_command("DMXP5#0#2")
# send_dmx_command("DMXC9#150#2")
# send_dmx_command("DMXP9#0")

# Prozentwert 55% auf Kanal 3, Geschwindigkeit 1
# send_dmx_command("DMXP4#55#1")

# # Prozentwert 56% auf Kanal 3, Geschwindigkeit 2
# send_dmx_command("DMXP3#56#2#2")

# # RGB-Wert: R=12, G=66, B=3 -> R + G*1000 + B*1000000
rgb_value = 80 + 5*1000 + 20*1000000  # = 3066012
send_dmx_command(f"DMXR9#{rgb_value}#2")
# send_dmx_command(f"DMXR10#{rgb_value}#2")
# send_dmx_command(f"DMXR10#{rgb_value}#1#1")

# # Tunable White (Typ V): 70% auf Kanal 5 (WW@5, CW@6)
# send_dmx_command("DMXV5#70")

# # Tunable White (Typ W): 70% auf Kanal 7 (CW@7, WW@8)

send_dmx_command("DMXW7#000128#3")
# send_dmx_command("DMXL1#20250030#3")

# # Du kannst auch eine Liste von Kommandos senden:
# commands = [
#     "DMXC1#255",   # Kanal 1 auf 255 setzen
#     "DMXP2#50",    # Kanal 2 auf 50%
#     "DMXR10#3066012",  # RGB auf Kanal 10
# ]


# for cmd in commands:
#     send_dmx_command(cmd)


Gesendet: DMXR9#20005080#2
Gesendet: DMXW7#000128#3


In [10]:
send_dmx_command("DMXL9#200802000#2") 


Gesendet: DMXL9#200802000#2


In [1]:
#Rest API Python Beispiel
import requests
import json

ESP32_IP = "udp2dmx2"  # <– hier deine ESP32-IP eintragen

# # Einzelwert ändern (PATCH):
# patch_data = {
#     "ct_config": {
#         "13": 2400,
#         "14": 6500
#     }
# }
# response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_data)
# print("PATCH Einzelwert:", response.status_code, response.text)

# # MinMax-Werte ändern (PATCH):
# patch_minmax = {
#     "default_ct": {
#         "min": 3000
#     }
# }
# response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_minmax)
# print("PATCH MinMax:", response.status_code, response.text)


# #Beispiel längere Json Datei
# patch_minmax = {
#     "ct_config": {
#         "1": 2500,
#         "7": 6000,
#         "8": 2000
#     },
#     "default_ct": {
#         "min": 3400,
#         "max": 6600
#     }
# }
patch_minmax = {
"hostname": "udp2dmx"
    }


response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_minmax)
print("PATCH MinMax:", response.status_code, response.text)



# # Ganze Datei lesen (GET):
# response = requests.get(f"http://{ESP32_IP}/config")
# print("GET Config:", response.status_code)
# print(response.json())

# # Ganze Datei ersetzen (POST):
# with open("main\components\spiffs_image\spiffs\config.json", "r") as f:
#     config_data = json.load(f)

# response = requests.post(f"http://{ESP32_IP}/config", json=config_data)
# print("POST Full Config:", response.status_code, response.text)


PATCH MinMax: 200 OK


In [11]:
#Beispielaufrufe für die REEST API:

# Einzelwert ändern:
curl -X POST http://<ESP32-IP>/config/patch \
     -H "Content-Type: application/json" \
     -d '{"ct_config": {"10": 5100}}'

# MinMax Werte ändern
curl -X POST http://<ESP32-IP>/config/patch \
     -H "Content-Type: application/json" \
     -d '{"default_ct": {"min": 3000}}'

# Ganze Datei lesen:
curl http://<ESP32-IP>/config

#Ganze Datei ersetzen/schreiben:
curl -X POST http://<ESP32-IP>/config \
     -H "Content-Type: application/json" \
     -d @ct_config.json

SyntaxError: invalid syntax (2666489373.py, line 4)

In [15]:
# === Mode-Tests (Kanäle 8–12) ===
# Unterstützte Modi im Firmware-Parser:
#   DMXC (0..255), DMXP (0..100%), DMXR (RGB, 3 Kanäle), DMXW (Tunable White, 2 Kanäle), DMXL (CT, 2 Kanäle)

import time

CHANNELS_1CH = list(range(8, 13))          # 8..12
CHANNELS_2CH = list(range(8, 12))          # 8..11 (braucht 2 Kanäle)
CHANNELS_3CH = list(range(8, 11))          # 8..10 (braucht 3 Kanäle)

# Loxone-speed: 255=sofort, sonst wird im ESP umgerechnet (siehe Logs für fade=ms)
SPEED_DIM = 210   # sichtbar dimmen (ca. ~720ms)
SPEED_STEP = 255  # sofort

def _send(cmd: str, pause_s: float = 0.25):
    send_dmx_command(cmd)
    time.sleep(pause_s)

def _rgb_value(r: int, g: int, b: int) -> int:
    # Firmware: r = value%1000, g=(value/1000)%1000, b=(value/1_000_000)%1000
    return int(r) + int(g) * 1000 + int(b) * 1_000_000

def _tw_value(ww: int, cw: int) -> int:
    # Firmware: ww=(value/1000)%1000, cw=value%1000  => value=ww*1000 + cw
    return int(ww) * 1000 + int(cw)

def _ct_value(brightness_percent: int, color_temp_k: int) -> int:
    # Firmware (DMXL): value 200000000..209999999
    # brightness = (value/10000)%1000 (0..100), ct=value%10000 (Kelvin)
    return 200_000_000 + int(brightness_percent) * 10_000 + int(color_temp_k)

print("== DMX Mode Test Start ==")

# 1) DMXC: Direktwerte (0..255) auf Kanäle 8..12
print("-- DMXC: Direktwerte --")
for ch in CHANNELS_1CH:
    for v in (0, 32, 128, 255, 0):
        _send(f"DMXC{ch}#{v}#{SPEED_STEP}")

# 2) DMXP: Prozent-Dimmen (0..100%) auf Kanäle 8..12
print("-- DMXP: Dimm-Sweep --")
for ch in CHANNELS_1CH:
    for pct in (0, 10, 25, 50, 75, 100, 75, 50, 25, 10, 0):
        _send(f"DMXP{ch}#{pct}#{SPEED_DIM}", pause_s=0.35)

# 3) DMXR: RGB Tests (Kanäle 8-10, 9-11, 10-12)
print("-- DMXR: RGB Farben --")
rgb_tests = [
    (255, 0, 0),
    (0, 255, 0),
    (0, 0, 255),
    (255, 255, 255),
    (0, 0, 0),
    (80, 5, 20),
    (0, 0, 0),
 ]
for start_ch in CHANNELS_3CH:
    for (r, g, b) in rgb_tests:
        _send(f"DMXR{start_ch}#{_rgb_value(r, g, b)}#{SPEED_DIM}", pause_s=0.4)

# 4) DMXW: Tunable White (WW/CW) auf Kanalpaaren 8..11
print("-- DMXW: Tunable White (WW/CW) --")
tw_tests = [
    (0, 0),
    (255, 0),
    (0, 255),
    (128, 128),
    (0, 0),
 ]
for start_ch in CHANNELS_2CH:
    for (ww, cw) in tw_tests:
        _send(f"DMXW{start_ch}#{_tw_value(ww, cw)}#{SPEED_DIM}", pause_s=0.4)

# 5) DMXL: CT (Helligkeit% + Kelvin) auf Kanalpaaren 8..11
print("-- DMXL: CT (Brightness% + Kelvin) --")
ct_tests = [
    (0, 2700),
    (25, 2700),
    (50, 4000),
    (75, 6500),
    (100, 6500),
    (0, 2700),
 ]
for start_ch in CHANNELS_2CH:
    for (bri, ct) in ct_tests:
        _send(f"DMXL{start_ch}#{_ct_value(bri, ct)}#{SPEED_DIM}", pause_s=0.45)

print("== DMX Mode Test Done ==")

== DMX Mode Test Start ==
-- DMXC: Direktwerte --
Gesendet: DMXC8#0#255
Gesendet: DMXC8#32#255
Gesendet: DMXC8#128#255
Gesendet: DMXC8#255#255
Gesendet: DMXC8#0#255
Gesendet: DMXC9#0#255
Gesendet: DMXC9#32#255
Gesendet: DMXC9#128#255
Gesendet: DMXC9#255#255
Gesendet: DMXC9#0#255
Gesendet: DMXC10#0#255
Gesendet: DMXC10#32#255
Gesendet: DMXC10#128#255
Gesendet: DMXC10#255#255
Gesendet: DMXC10#0#255
Gesendet: DMXC11#0#255
Gesendet: DMXC11#32#255
Gesendet: DMXC11#128#255
Gesendet: DMXC11#255#255
Gesendet: DMXC11#0#255
Gesendet: DMXC12#0#255
Gesendet: DMXC12#32#255
Gesendet: DMXC12#128#255
Gesendet: DMXC12#255#255
Gesendet: DMXC12#0#255
-- DMXP: Dimm-Sweep --
Gesendet: DMXP8#0#210
Gesendet: DMXP8#10#210
Gesendet: DMXP8#25#210
Gesendet: DMXP8#50#210
Gesendet: DMXP8#75#210
Gesendet: DMXP8#100#210
Gesendet: DMXP8#75#210
Gesendet: DMXP8#50#210
Gesendet: DMXP8#25#210
Gesendet: DMXP8#10#210
Gesendet: DMXP8#0#210
Gesendet: DMXP9#0#210
Gesendet: DMXP9#10#210
Gesendet: DMXP9#25#210
Gesendet: DMXP9#5

# OTA Upload (Firmware & Web-UI)

> Diese Zellen laden die Artefakte aus dem Repo-Buildordner hoch:
- Firmware: `build/UDP2DMX.bin` → `POST http://<ESP>/api/ota/upload`
- Web-UI (SPIFFS): `build/spiffs.bin` → `POST http://<ESP>/api/ota/spiffs`

> Hinweis: Nach erfolgreichem Upload rebootet das Gerät i. d. R. automatisch (je nach Endpoint/Implementierung).

In [9]:
import time
import threading
from pathlib import Path

import requests

# ==================== Settings ====================
# Ziel-Host/IP deines ESP32 (mDNS-Name oder IP)
ESP32_HOST = "192.168.178.57"  # z.B. "udp2dmx.local" oder "192.168.188.85"

# Standard-Artefakte aus dem Build
FIRMWARE_BIN = Path("build") / "UDP2DMX.bin"
SPIFFS_BIN   = Path("build") / "spiffs.bin"

# Harte Obergrenze, damit Upload nie „ewig“ hängt (Sekunden)
# (Upload kann bei langsamen WLANs dauern → bei Bedarf erhöhen)
UPLOAD_HARD_TIMEOUT_S = 240

# ==================== Helpers ====================
def _url(host: str, path: str) -> str:
    return f"http://{host}{path}"

def _assert_file(p: Path):
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p.resolve()}")
    if p.stat().st_size == 0:
        raise ValueError(f"File is empty: {p.resolve()}")

def wait_for_http(host: str, path: str = "/api/system", timeout_s: int = 60, interval_s: float = 1.0):
    """Wait until HTTP endpoint is reachable again (after reboot)."""
    deadline = time.time() + timeout_s
    last_err = None
    while time.time() < deadline:
        try:
            r = requests.get(_url(host, path), timeout=5)
            if r.ok:
                return r
            last_err = f"HTTP {r.status_code}"
        except Exception as e:
            last_err = str(e)
        time.sleep(interval_s)
    raise TimeoutError(f"ESP not reachable on {host}{path} within {timeout_s}s (last: {last_err})")

def _post_file_no_hang(
    url: str,
    file_path: Path,
    timeout_s: int,
    content_type: str = "application/octet-stream",
    ):
    """
    POST file contents, but never hang indefinitely.

    Some OTA handlers reboot quickly and may not send a response (or cut the TCP connection).
    To avoid the notebook waiting forever on the HTTP client, this runs the POST in a daemon
    thread and returns `None` if it exceeds `timeout_s`.
    """
    result = {"resp": None, "exc": None}

    def _worker():
        try:
            with open(file_path, "rb") as f:
                # timeout=(connect, read) – socket timeout usually applies to send/recv too,
                # but we still protect with the outer thread hard-timeout.
                result["resp"] = requests.post(
                    url,
                    data=f,
                    headers={
                        "Content-Type": content_type,
                        "Connection": "close",
                    },
                    timeout=(10, timeout_s),
                )
        except Exception as e:
            result["exc"] = e

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join(timeout_s)

    if t.is_alive():
        # Hard stop: we do NOT wait longer (thread continues as daemon, but notebook continues).
        print(f"⚠️ No HTTP response within {timeout_s}s (likely reboot or stalled connection).")
        return None

    if result["exc"] is not None:
        raise result["exc"]

    return result["resp"]

def ota_upload_firmware(host: str, bin_path: Path = FIRMWARE_BIN, timeout_s: int = UPLOAD_HARD_TIMEOUT_S):
    """Upload Firmware (.bin) via OTA endpoint (device will reboot)."""
    _assert_file(bin_path)
    url = _url(host, "/api/ota/upload")
    print("Uploading firmware:", bin_path)
    print(" ->", url)
    try:
        resp = _post_file_no_hang(url, bin_path, timeout_s=timeout_s)
        if resp is None:
            print("(no response) – assuming firmware upload finished and device rebooted")
            return None
        print("HTTP", resp.status_code)
        print(resp.text[:2000])
        # If the device responded, still validate success code
        resp.raise_for_status()
        return resp
    except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
        # Typical when the device reboots before replying / connection is cut
        print(f"(connection ended: {e}) – assuming reboot")
        return None

def ota_upload_spiffs(host: str, bin_path: Path = SPIFFS_BIN, timeout_s: int = UPLOAD_HARD_TIMEOUT_S):
    """Upload SPIFFS/Web-UI image (.bin) via OTA endpoint (device will reboot)."""
    _assert_file(bin_path)
    url = _url(host, "/api/ota/spiffs")
    print("Uploading SPIFFS:", bin_path)
    print(" ->", url)
    try:
        resp = _post_file_no_hang(url, bin_path, timeout_s=timeout_s)
        if resp is None:
            print("(no response) – assuming SPIFFS upload finished and device rebooted")
            return None
        print("HTTP", resp.status_code)
        print(resp.text[:2000])
        resp.raise_for_status()
        return resp
    except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
        print(f"(connection ended: {e}) – assuming reboot")
        return None

def ota_upload_spiffs_and_firmware(
    host: str = ESP32_HOST,
    spiffs_path: Path = SPIFFS_BIN,
    firmware_path: Path = FIRMWARE_BIN,
    reboot_wait_s: int = 120,
    settle_s: float = 2.0,
    timeout_s: int = UPLOAD_HARD_TIMEOUT_S,
    verify: bool = True,
    ):
    """
    Lädt zuerst SPIFFS (Web-UI) und danach Firmware hoch.

    Wichtig: Beide Upload-Endpunkte rebooten das Gerät. Daher wartet diese Funktion
    zwischen den Uploads jeweils, bis `/api/system` wieder erreichbar ist.
    """
    print("ESP:", host)
    print("Firmware:", firmware_path, "exists=", firmware_path.exists())
    print("SPIFFS:", spiffs_path, "exists=", spiffs_path.exists())

    if verify:
        print("Checking device reachability...")
        wait_for_http(host, timeout_s=reboot_wait_s)

    # 1) SPIFFS (reboot)
    ota_upload_spiffs(host, spiffs_path, timeout_s=timeout_s)
    time.sleep(settle_s)
    print("Waiting after SPIFFS reboot...")
    wait_for_http(host, timeout_s=reboot_wait_s)

    # 2) Firmware (reboot)
    ota_upload_firmware(host, firmware_path, timeout_s=timeout_s)
    time.sleep(settle_s)
    print("Waiting after firmware reboot...")
    wait_for_http(host, timeout_s=reboot_wait_s)

    print("✅ Done: SPIFFS + Firmware uploaded")

# Ausführen:
ota_upload_spiffs_and_firmware()

ESP: 192.168.178.57
Firmware: build\UDP2DMX.bin exists= True
SPIFFS: build\spiffs.bin exists= True
Checking device reachability...
Uploading SPIFFS: build\spiffs.bin
 -> http://192.168.178.57/api/ota/spiffs
HTTP 200
OK - rebooting
Waiting after SPIFFS reboot...
Uploading firmware: build\UDP2DMX.bin
 -> http://192.168.178.57/api/ota/upload
HTTP 200
OK - rebooting
Waiting after firmware reboot...
✅ Done: SPIFFS + Firmware uploaded
